# Soccer Vision v0.1 - Interactive Debugging Notebook

This notebook is the primary way to develop and debug the pipeline. Each cell does one focused thing. Run them top to bottom on first use, then iterate on whichever cell you're tuning.

**Workflow:**
1. Setup + load a video
2. Run the pipeline on a few frames and inspect the raw Python objects
3. Render an annotated mp4 you can play inline
4. Tune thresholds and re-run
5. Look at frame-level stats (touches, ball detection rate)

## 1. Setup

If you used the `environment.yml` to create the conda env, everything is already installed. The first run will auto-download the YOLOv8n weights (~6 MB) into `~/.cache/ultralytics/`.

In [ ]:
# This makes editing soccer_vision/*.py reload without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make the soccer_vision package importable from this notebook.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from soccer_vision.pipeline import Pipeline
from soccer_vision.viz import draw_overlay, render_to_mp4
print('Imports OK. Repo root:', REPO_ROOT)

## 2. Pick a test clip

Drop a short clip (~30 seconds) into `data/`. If you don't have one, see the README for how to grab a SoccerNet sample with `yt-dlp`.

In [ ]:
VIDEO_PATH = REPO_ROOT / 'data' / 'test_clip.mp4'
assert VIDEO_PATH.exists(), f'Put a clip at {VIDEO_PATH} first'
VIDEO_PATH

## 3. Run the pipeline on the first 30 frames and inspect

Don't render video yet - just look at what the model produced as Python objects. This is faster to iterate on and easier to debug.

In [ ]:
pipe = Pipeline(
    detection_confidence=0.30,
    ball_detection_confidence=0.10,  # ball is hard, lower the bar
    sample_every_n=3,                # 30fps source -> 10fps processing
    proximity_threshold_ratio=0.6,   # ball within 0.6x bbox diagonal = proximity
)

# Run on first 30 sampled frames and keep results in memory
results = []
frames = []
for frame, result in pipe.run(str(VIDEO_PATH), max_frames=30):
    frames.append(frame)
    results.append(result)

print(f'Got {len(results)} frame results')

In [ ]:
# Quick numeric overview - is the model doing anything?
import statistics

player_counts = [len(r.players) for r in results]
ball_hits = sum(1 for r in results if r.ball is not None)
proximity_counts = [len(r.proximity_events) for r in results]

print(f'Players per frame: min={min(player_counts)} '
      f'med={statistics.median(player_counts):.0f} '
      f'max={max(player_counts)}')
print(f'Ball detected in {ball_hits}/{len(results)} frames '
      f'({100 * ball_hits / len(results):.0f}%)')
print(f'Proximity events per frame (avg): '
      f'{statistics.mean(proximity_counts):.2f}')

In [ ]:
# Inspect one specific frame's objects directly
r = results[0]
print(f'Frame {r.frame_index} @ {r.timestamp_ms}ms')
print(f'  raw detections: {len(r.raw_detections)}')
print(f'  tracked players: {len(r.players)}')
for p in r.players[:5]:
    print(f'    {p.track_id}  bbox={tuple(round(v,1) for v in p.bbox_xyxy)}  conf={p.confidence:.2f}')
print(f'  ball: {r.ball}')
print(f'  nearest player to ball: {r.nearest_player_to_ball}')

## 4. Visual sanity check - one frame with overlay

Pick a frame where the ball was detected and look at it.

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Find a frame with both players and ball detected
good_idx = next((i for i, r in enumerate(results) 
                 if r.ball is not None and len(r.players) > 5), 0)
annotated = draw_overlay(frames[good_idx], results[good_idx])

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f'Frame {results[good_idx].frame_index}: green=player, orange=ball, red=nearest player to ball')
plt.show()

## 5. Render an annotated video and play it inline

This processes the clip, draws boxes on every frame, saves an mp4, and embeds it in the notebook so you can scrub through it.

On a CPU laptop, expect ~1-3 frames per second of throughput. A 30-second clip at 10fps = 300 frames, so 2-5 minutes of wall time.

In [ ]:
OUT_PATH = REPO_ROOT / 'data' / 'annotated.mp4'

# Fresh pipeline so trackers reset
pipe = Pipeline(
    detection_confidence=0.30,
    ball_detection_confidence=0.10,
    sample_every_n=3,
    proximity_threshold_ratio=0.6,
)

# Render the entire clip (no max_frames)
from tqdm.notebook import tqdm

def progress_iter(gen):
    pbar = tqdm(desc='Processing', unit='frame')
    for item in gen:
        pbar.update(1)
        yield item
    pbar.close()

msg = render_to_mp4(progress_iter(pipe.run(str(VIDEO_PATH))), str(OUT_PATH), fps=10)
print(msg)

In [ ]:
# Embed the annotated video in the notebook for inline playback.
# Click the play button on the player below.
from IPython.display import Video
Video(str(OUT_PATH), embed=True, width=900)

## 6. Per-player proximity timeline

For each tracked player, count how many frames they were nearest to the ball. This is the closest thing v0.1 has to a 'touches' stat. Treat it as a possession proxy, not ground truth - see the caveats in `types.py:ProximityEvent`.

In [ ]:
from collections import Counter

# Re-run on the whole clip if we don't already have full results
if len(results) < 30:
    results = []
    pipe = Pipeline(detection_confidence=0.30, ball_detection_confidence=0.10,
                    sample_every_n=3, proximity_threshold_ratio=0.6)
    for _, r in pipe.run(str(VIDEO_PATH)):
        results.append(r)

nearest_counts = Counter()
for r in results:
    np_ = r.nearest_player_to_ball
    if np_ is not None and np_.bbox_intersects:
        nearest_counts[np_.track_id] += 1

print('Frames with bbox intersection (proxy for touches), by player:')
for tid, n in nearest_counts.most_common(10):
    print(f'  {tid}: {n} frames')

## 7. Tweak parameters

Try changing one thing at a time:

- `detection_confidence=0.20` - more players, more noise
- `ball_detection_confidence=0.05` - more ball hits, more false positives
- `proximity_threshold_ratio=1.0` - more proximity events (looser)
- `sample_every_n=1` - process every frame, much slower

Re-run cell 5 to see the visual difference.